# Wang 2020 VLST integer score on `VLST.csv`

Reconstructs the **published 8-variable points score** (Wang et al., *Sci Rep* 2020;10:6378, Table 2) on this
repository’s derivation file. The score is **frozen** — weights are not re-fit. Part 4 Table S-Wang is the write-up
(evidence map §5.10 / B10 closed for this comparator).

| Variable (Wang Table 2) | Points | Column in `VLST.csv` |
| --- | ---: | --- |
| Diabetes mellitus | 1 | `Diabetes` |
| Previous PCI | 3 | `Previous PCI` |
| AMI as admitting diagnosis | 1 | `Initial diagnosis-AMI` |
| eGFR < 90 | 1 | `CKD90` (= `eGFR < 90`) |
| 3-vessel disease | 1 | `3-vessel disease` |
| No. of stents per lesion | 2 × count | `No.of stents per lesion` |
| Stent type–SES | 1 | **`PES`** (counts match Wang Table 1 SES 82.61% / 68.76%) |
| No post-dilation | 4 | **`No postdilation`** (risk direction; 78/92 events) |

**Label caveats (do not copy Wang Table 1 blindly).**

1. Wang Table 1 “No post-dilation” in 14/92 VLST cases is this file’s `1.1:1Post dilation` = 1. Table 2 still
   assigns **4 risk points** and β = 1.93, while the printed HR 0.145 is `exp(−1.93)`. Using those 14 cases as
   the 4-point group yields ROC-AUC ≈ 0.51 (anti-predictive). Using `No postdilation` = 1 reproduces c ≈ 0.80
   and the published high-risk bin n = 473.
2. Wang Table 1 “SES” rates are identical to this file’s **`PES` flag**, not the 106-level `Stent type-SES`
   strings. The score uses `PES` so the published split is recovered.

Shantou (n = 2,058) is not in the repo. This notebook is derivation-cohort scoring only.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / "data" / "raw" / "VLST.csv").is_file()),
    HERE,
)
sys.path.insert(0, str(ROOT / "code" / "modeling" / "tools"))
from figure_style import HARMONY, apply_style  # noqa: E402

apply_style()

OUT_DIRS = [
    ROOT / "paper_results" / "04_tabpfn_rating" / "paper_figures",
    ROOT / "code" / "modeling" / "rating" / "paper_figures",
]
CSV_PATH = ROOT / "data" / "raw" / "VLST.csv"
print("ROOT", ROOT)
print("CSV", CSV_PATH)


In [ ]:
# Published Table 2 integer weights (Wang 2020).
WANG_POINTS = {
    "Diabetes": 1,
    "Previous PCI": 3,
    "Initial diagnosis-AMI": 1,
    "CKD90": 1,
    "3-vessel disease": 1,
    "No.of stents per lesion": 2,  # multiplied by count
    "PES": 1,  # Wang Table 1 "SES" counts
    "No postdilation": 4,  # risk direction in CSV
}

df = pd.read_csv(CSV_PATH)
y = df["Stent thrombosis"].astype(int)
assert len(df) == 5185 and int(y.sum()) == 92

# Encoding checks
assert (df["eGFR"] < 90).astype(int).eq(df["CKD90"].astype(int)).all()
assert df["No postdilation"].eq(1 - df["1.1:1Post dilation"]).all()
pes_vlst = float(df.loc[y == 1, "PES"].mean())
pes_ctrl = float(df.loc[y == 0, "PES"].mean())
assert abs(pes_vlst - 0.8261) < 1e-3
assert abs(pes_ctrl - 0.6876) < 1e-3

score = (
    WANG_POINTS["Diabetes"] * df["Diabetes"]
    + WANG_POINTS["Previous PCI"] * df["Previous PCI"]
    + WANG_POINTS["Initial diagnosis-AMI"] * df["Initial diagnosis-AMI"]
    + WANG_POINTS["CKD90"] * df["CKD90"].astype(int)
    + WANG_POINTS["3-vessel disease"] * df["3-vessel disease"]
    + WANG_POINTS["No.of stents per lesion"] * df["No.of stents per lesion"].astype(int)
    + WANG_POINTS["PES"] * df["PES"]
    + WANG_POINTS["No postdilation"] * df["No postdilation"]
).astype(int)

# Wrong post-dilation polarity (Wang Table 1's 14 VLST cases).
score_flipped = score - 4 * df["No postdilation"] + 4 * df["1.1:1Post dilation"]

roc = roc_auc_score(y, score)
ap = average_precision_score(y, score)
roc_flip = roc_auc_score(y, score_flipped)
print(f"Frozen Wang integer score: ROC-AUC={roc:.4f}  PR-AUC={ap:.4f}")
print(f"Flipped 4-pt post-dilation (Table 1 label): ROC-AUC={roc_flip:.4f}")
print("Wang published derivation c-statistic = 0.80 (95% CI 0.75–0.85)")


In [ ]:
bins = pd.cut(score, bins=[-np.inf, 7, 9, np.inf], labels=["low (≤7)", "intermediate (8–9)", "high (≥10)"])
bin_rows = []
for label in ["low (≤7)", "intermediate (8–9)", "high (≥10)"]:
    m = bins == label
    bin_rows.append({
        "Risk category": label,
        "n": int(m.sum()),
        "% of cohort": round(100 * m.mean(), 1),
        "VLST events": int(y[m].sum()),
        "Observed VLST rate": round(float(y[m].mean()), 4),
        "Wang published n": {"low (≤7)": 3135, "intermediate (8–9)": 1837, "high (≥10)": 473}[label],
        "Wang published rate": {"low (≤7)": 0.005, "intermediate (8–9)": 0.022, "high (≥10)": 0.087}[label],
    })
bin_df = pd.DataFrame(bin_rows)
print(bin_df.to_string(index=False))
print()
print("Wang's three published n's sum to 5445, not 5185; low=3135 and high=473 match this file exactly.")
print("Intermediate remainder here is 1577 (rate 0.0222, matching Wang's 2.2%).")

# Same outer folds as Part 4 nested CV (frozen score: evaluate only).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_rows = []
for fold, (_, val_idx) in enumerate(cv.split(df, y), 1):
    yt, s = y.iloc[val_idx], score.iloc[val_idx]
    fold_rows.append({
        "fold": fold,
        "n": int(len(val_idx)),
        "events": int(yt.sum()),
        "ROC-AUC": roc_auc_score(yt, s),
        "PR-AUC": average_precision_score(yt, s),
    })
fold_df = pd.DataFrame(fold_rows)
print("\nPart 4 outer folds (score not refit):")
print(fold_df.round(4).to_string(index=False))
print(
    f"fold mean ROC-AUC {fold_df['ROC-AUC'].mean():.4f} ± {fold_df['ROC-AUC'].std(ddof=1):.4f}  |  "
    f"PR-AUC {fold_df['PR-AUC'].mean():.4f} ± {fold_df['PR-AUC'].std(ddof=1):.4f}"
)

compare = pd.DataFrame([
    {
        "Model": "Wang 2020 integer score (frozen)",
        "ROC-AUC": round(roc, 4),
        "PR-AUC": round(ap, 4),
        "ROC fold mean ± SD": f"{fold_df['ROC-AUC'].mean():.4f} ± {fold_df['ROC-AUC'].std(ddof=1):.4f}",
        "PR fold mean ± SD": f"{fold_df['PR-AUC'].mean():.4f} ± {fold_df['PR-AUC'].std(ddof=1):.4f}",
        "Protocol": "Published points on all 5,185 rows; folds are evaluation only",
    },
    {
        "Model": "LightGBM (untuned nested CV)",
        "ROC-AUC": 0.9681,
        "PR-AUC": 0.6937,
        "ROC fold mean ± SD": "0.9695 ± 0.0164",
        "PR fold mean ± SD": "0.6941 ± 0.0917",
        "Protocol": "Part 4 nested 5×4 CV OOF (this notebook)",
    },
    {
        "Model": "TabPFN (local)",
        "ROC-AUC": 0.9845,
        "PR-AUC": 0.6754,
        "ROC fold mean ± SD": "0.9846 ± 0.0030",
        "PR fold mean ± SD": "0.6739 ± 0.0812",
        "Protocol": "Part 4 nested 5×4 CV OOF (this notebook)",
    },
    {
        "Model": "Logistic regression (untuned nested CV)",
        "ROC-AUC": 0.9224,
        "PR-AUC": 0.3326,
        "ROC fold mean ± SD": "0.9235 ± 0.0251",
        "PR fold mean ± SD": "0.3451 ± 0.1213",
        "Protocol": "Part 4 nested 5×4 CV OOF (this notebook)",
    },
])
print("\nComparator (D4: LightGBM / TabPFN local / LR from executed Part 4 notebook):")
print(compare.to_string(index=False))


In [ ]:
def _save(fig, name: str) -> None:
    for out in OUT_DIRS:
        out.mkdir(parents=True, exist_ok=True)
        fig.savefig(out / name, dpi=300, bbox_inches="tight", facecolor="white")


def _write_csv(frame: pd.DataFrame, name: str) -> None:
    for out in OUT_DIRS:
        out.mkdir(parents=True, exist_ok=True)
        frame.to_csv(out / name, index=False)


def _table_image(frame: pd.DataFrame, name: str) -> None:
    n_rows, n_cols = frame.shape
    fig, ax = plt.subplots(figsize=(min(16, 1.1 * n_cols + 6), max(2.2, 0.42 * (n_rows + 2))))
    ax.axis("off")
    tbl = ax.table(
        cellText=frame.astype(str).values,
        colLabels=list(frame.columns),
        loc="center",
        cellLoc="left",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 1.3)
    for (r, _), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(HARMONY[7])
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#F4F7FA")
    _save(fig, name)
    plt.close(fig)


_write_csv(bin_df, "paper_table_s_wang_score_bins.csv")
_table_image(bin_df, "paper_table_s_wang_score_bins.png")
_write_csv(compare, "paper_table_s_wang_vs_ml.csv")
_table_image(compare, "paper_table_s_wang_vs_ml.png")
_write_csv(fold_df.round(4), "paper_table_s_wang_score_folds.csv")

# Figure: observed VLST rate by integer score
rates, ns, xs = [], [], []
for s in range(int(score.min()), int(score.max()) + 1):
    m = score == s
    if m.sum() == 0:
        continue
    xs.append(s)
    ns.append(int(m.sum()))
    rates.append(float(y[m].mean()))

fig, ax = plt.subplots(figsize=(8.2, 4.6))
ax.bar(xs, rates, color=HARMONY[7], width=0.8, label="Observed VLST rate")
ax.axhline(float(y.mean()), color=HARMONY[0], ls="--", lw=1.2, label=f"Cohort prevalence {y.mean():.4f}")
ax.set_xlabel("Wang 2020 integer score (frozen)")
ax.set_ylabel("Observed VLST rate")
ax.set_title("Derivation cohort: VLST rate by published score")
ax.legend(frameon=False)
for x, n, r in zip(xs, ns, rates):
    if n >= 20:
        ax.text(x, r + 0.004, str(n), ha="center", va="bottom", fontsize=7, color="#444")
_save(fig, "paper_fig_s_wang_score_rate.png")
plt.close(fig)

print("Wrote:")
for d in OUT_DIRS:
    print(" ", d)
